# 1. Exploratory Data Analysis (EDA)

## 1.1 Import Libraries and Load Data
First, I'll import the necessary libraries and load my dataset from the `data.csv` file.

In [ ]:
# Import libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load the dataset from the CSV file
df = pd.read_csv('data.csv')

# Show the first 5 rows to check if it loaded
df.head()

## 1.2 Check Data Structure & Missing Values

Here, I'll run `df.info()` to see the data types of each column (like text or number) and to find out how many values are missing.

In [ ]:
# Get a summary of all columns (types, missing values)
df.info()

## 1.3 Check My Target Variable (Loan_Status)

Now I'll count the 'Y' (Yes) and 'N' (No) values in my target column (`Loan_Status`) to see if the dataset is imbalanced.

In [ ]:
# Count the 'Y' and 'N' values in our target column
print(df['Loan_Status'].value_counts())

# Plot the target variable
sns.countplot(x='Loan_Status', data=df)
plt.show()

## 1.4 Check Numeric Columns

This gives me a quick statistical summary (like mean, min, max) for all columns that contain numbers.

In [ ]:
# Get quick stats (mean, min, max) for number columns
df.describe()

## 1.5 Check Categorical (Text) Columns

Finally, I'll look at the different categories in my main text-based columns.

In [ ]:
# Check unique values in a few text columns
print("--- Gender Counts ---")
print(df['Gender'].value_counts())

print("\n--- Education Counts ---")
print(df['Education'].value_counts())

print("\n--- Property_Area Counts ---")
print(df['Property_Area'].value_counts())

# 2. Data Cleaning & Pre-processing

Now that I've explored the data, I need to clean it.
My plan is to:
1.  Fill in all the missing values (imputation).
2.  Convert all text columns (categorical) into numbers (encoding).

## 2.1 Handle Missing Values

Based on my `df.info()` output, I have missing values in several columns. I'll fill them in using the most logical value.
* For number columns like `LoanAmount`, I'll use the **median** (the middle value) because the `df.describe()` output showed it has outliers.
* For text columns like `Gender` or `Married`, I'll use the **mode** (the most common value).

In [ ]:
# --- Fill Missing Numeric Values ---
# Fill missing 'LoanAmount' with the median (middle) value
df['LoanAmount'] = df['LoanAmount'].fillna(df['LoanAmount'].median())

# Fill missing 'Loan_Amount_Term' with the mode (most common) value
df['Loan_Amount_Term'] = df['Loan_Amount_Term'].fillna(df['Loan_Amount_Term'].mode()[0])

# Fill missing 'Credit_History' with the mode (most common) value (which is 1.0)
df['Credit_History'] = df['Credit_History'].fillna(df['Credit_History'].mode()[0])

# --- Fill Missing Categorical (Text) Values ---
# Fill 'Gender', 'Married', 'Dependents', and 'Self_Employed' with the mode
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])
df['Married'] = df['Married'].fillna(df['Married'].mode()[0])
df['Dependents'] = df['Dependents'].fillna(df['Dependents'].mode()[0])
df['Self_Employed'] = df['Self_Employed'].fillna(df['Self_Employed'].mode()[0])

# --- Check if all missing values are fixed ---
print("--- Missing Values After Cleaning ---")
print(df.isnull().sum())

## 2.2 Convert Categorical (Text) Data to Numbers

My model can't understand text like 'Male' or 'Y'. I need to convert these into numbers.
* `Loan_Status` (my target) will become: `Y`=1, `N`=0.
* Other columns like `Gender` will become: `Male`=1, `Female`=0.
* `Dependents` and `Property_Area` will also be converted.

In [ ]:
# --- Convert Target Variable ---
# Change 'Y' and 'N' in 'Loan_Status' to 1 and 0
df['Loan_Status'] = df['Loan_Status'].map({'Y': 1, 'N': 0})

# --- Convert Other Categorical Columns ---
# Change 'Gender' to 1 (Male) and 0 (Female)
df['Gender'] = df['Gender'].map({'Male': 1, 'Female': 0})

# Change 'Married' to 1 (Yes) and 0 (No)
df['Married'] = df['Married'].map({'Yes': 1, 'No': 0})

# Change 'Education' to 1 (Graduate) and 0 (Not Graduate)
df['Education'] = df['Education'].map({'Graduate': 1, 'Not Graduate': 0})

# Change 'Self_Employed' to 1 (Yes) and 0 (No)
df['Self_Employed'] = df['Self_Employed'].map({'Yes': 1, 'No': 0})

# 'Dependents' has '3+' which isn't a number. We'll replace '3+' with 3.
df['Dependents'] = df['Dependents'].replace('3+', '3')
# Now convert the whole column to a number
df['Dependents'] = pd.to_numeric(df['Dependents'])

# 'Property_Area' has 3 values. We'll use get_dummies to convert it
# This creates 3 new columns (Urban, Rural, Semiurban) with 1s and 0s
df = pd.get_dummies(df, columns=['Property_Area'])

# --- Check our work ---
print("\n--- Data Head After Encoding ---")
df.head()

## 2.3 Final Check

Now I'll run `df.info()` one last time to confirm all data is numeric and there are no missing values.

In [ ]:
# --- Final Check ---
# All columns should now have 614 non-null values
# All columns should be a number (int64, float64, or uint8)
df.info()

# 3. Model Preparation & Training

## 3.1 Define Features (X) and Target (y)
First, I separate the features from the target variable.

In [ ]:
# Define Target (y) and Features (X)
y = df['Loan_Status']
X = df.drop(['Loan_Status', 'Loan_ID'], axis=1)

## 3.2 Split Data (The "Exam" Separation)
I split the data **BEFORE** scaling to prevent data leakage.

In [ ]:
from sklearn.model_selection import train_test_split

# SPLIT FIRST!
# We split into Train (80%) and Test (20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data Split Successfully.")
print(f"Training Rows: {X_train.shape[0]}")
print(f"Testing Rows: {X_test.shape[0]}")

## 3.3 Scale the Data (Correctly)
Now that the data is split, I will fit the scaler **ONLY on the Training data**.
Then I will apply that same ruler to the Test data.

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Create the Scaler
scaler = StandardScaler()

# 2. Fit ONLY on Training Data (Learn the mean/std from the study guide only)
scaler.fit(X_train)

# 3. Transform BOTH sets
# We use the scaler we just trained to transform X_train and X_test
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data Scaled Successfully (No Leakage).")

## 3.4 Train the Model
Now I train the Logistic Regression model using the **scaled training data**.

In [ ]:
from sklearn.linear_model import LogisticRegression

# Create and Train the Model
model_logreg = LogisticRegression(max_iter=1000)
model_logreg.fit(X_train_scaled, y_train)

print("Logistic Regression Model Trained.")

# 4. Model Evaluation

Now that the model is trained, I need to evaluate its performance using the **Test Set**.
I will use the **scaled test data** (`X_test_scaled`) to make predictions, and then compare those predictions to the real answers (`y_test`).

In [ ]:
# --- 1. Import Evaluation Metrics ---
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# --- 2. Make Predictions ---
# **CRITICAL:** We use the SCALED test data (X_test_scaled) because we trained on scaled data.
# If we used the unscaled 'X_test', the predictions would be complete garbage.
y_pred_logreg = model_logreg.predict(X_test_scaled)

# --- 3. Evaluate the Predictions ---

# A. Check the Accuracy (How often is it right?)
accuracy = accuracy_score(y_test, y_pred_logreg)
print(f"--- Logistic Regression Accuracy ---")
print(f"Accuracy: {accuracy * 100:.2f}%")

# B. Show the Confusion Matrix (Where did it make mistakes?)
# [True Negatives  False Positives]
# [False Negatives True Positives]
print("\n--- Confusion Matrix ---")
cm = confusion_matrix(y_test, y_pred_logreg)
print(cm)

# C. Show the Classification Report (Precision, Recall, F1-Score)
# This gives us a detailed score for both 'Yes' and 'No' predictions.
print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred_logreg))

# 5. Model Analysis & Interpretability

One major advantage of Logistic Regression is that it is **interpretable**. “Interpretable” means you can explain the model’s decisions in a way humans understand.
I will extract the model's **coefficients** to see which features (like Credit History or Income) have the biggest impact on loan approval.